# Comparative Requirements Extraction Analysis
The purpose of this notebook is to compare LLM extracted/generated US Core (Version 8.0.0) requirements against manually generated requirements. 

Importing libraries

In [1]:
# %pip install openpyxl

In [4]:
import pandas as pd
import logging
from pathlib import Path
from typing import Optional, List
import tqdm as notebook_tqdm
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

Loading data

In [5]:
INPUT_FILE = "data/hl7.fhir.us.core_8.0.0_reqs.xlsx"
TARGET_COLUMNS = ['ID*', 'Requirement*', 'Conformance*', 'Actors*', 'Conditionality']

In [8]:
us_core_manual_reqs = pd.read_excel(INPUT_FILE, sheet_name='Requirements', engine='openpyxl')

/Users/amathur/Library/Python/3.9/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [11]:
us_core_llm_reqs = pd.read_csv("data/us.core_8.0.0_reqs_llm.csv")

Inspecting data

In [25]:
# Inspect columns to identify where requirements are stored
# us_core_llm_reqs.head()
# us_core_manual_reqs.head()

Choosing requirement text columns from each dataframe

In [14]:
us_core_llm_reqs_text = us_core_llm_reqs["summary"].astype(str).tolist()

In [16]:
us_core_manual_reqs_text = us_core_manual_reqs["Requirement*"].astype(str).tolist()

Initialize Embeddings Model

Using HuggingFace SBERT model wraped for LangChain.

In [17]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/f6/7_jgcr3x6tjggn2mh6nvh3z00000gn/T/ipykernel_8245/412152783.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Building FAISS Index (vector store) from US Core Manual requirements

In [18]:
us_core_manual_reqs_store = FAISS.from_texts(us_core_manual_reqs_text, embedding_model)

Now for each requirement in LLM US Core data, find the top N most similar requirements from the US Core manual data

In [19]:
results = []

In [20]:
for i, req in enumerate(us_core_llm_reqs_text):
    matches = us_core_manual_reqs_store.similarity_search_with_score(req, k=3)  # top-3 matches with scores
    for m, score in matches:
        results.append({
            "us_core_llm_req_id": i,
            "us_core_llm_req": req,
            "us_core_manual_req_match": m.page_content,
            "similarity_score": score
        })

In [21]:
results_df = pd.DataFrame(results)

NOTES ON SCORES:
the score is actually the cosine distance, so lower score = more similar

In [22]:
results_df.head(5)

,us_core_llm_req_id,us_core_llm_req,us_core_manual_req_match,similarity_score
0,0,USCoreCareplanStatus SearchParameter shall not...,When searching using the `token` type searchpa...,0.878464
1,0,USCoreCareplanStatus SearchParameter shall not...,US Core SearchParameters referenced in [the US...,0.878861
2,0,USCoreCareplanStatus SearchParameter shall not...,US Core SearchParameters referenced in [the US...,0.893511
3,1,Servers and Clients should use standard FHIR S...,Clients **SHOULD** use the standard FHIR Searc...,0.445147
4,1,Servers and Clients should use standard FHIR S...,Servers ... **SHOULD** use the standard FHIR S...,0.538727


We can sort the dataframe to see the most similar requirements emerge to the top.

In [23]:
results_df_sorted = results_df.sort_values(by="similarity_score", ascending=True)

In [24]:
results_df_sorted

,us_core_llm_req_id,us_core_llm_req,us_core_manual_req_match,similarity_score
1698,566,Implantable medical devices with UDI informati...,Implantable medical devices with UDI informati...,0.002856
1407,469,Implantable medical devices with UDI informati...,Implantable medical devices with UDI informati...,0.002856
3420,1140,Systems should support Clinical Laboratory Imp...,Systems … SHOULD support Clinical Laboratory I...,0.012565
3417,1139,Systems must support National Provider Identif...,Systems SHALL support National Provider Identi...,0.024581
3912,1304,"When recording self-prescribed medication, req...","When recording “self-prescribed” medication, r...",0.038076
...,...,...,...,...
438,146,Normal processing when no adjustments needed,[When using US Core Treatment Intervention Pre...,1.498294
2321,773,whenHandedOver cannot be before whenPrepared,The Client application SHALL support ... Goal....,1.500022
439,146,Normal processing when no adjustments needed,Systems **SHOULD** designate the patient's pre...,1.511404
440,146,Normal processing when no adjustments needed,The `Procedure.performed` is mandatory if `Pro...,1.516169
